# HC3 Dataset Preprocessing Pipeline

This notebook prepares the HC3 (Human ChatGPT Comparison Corpus) dataset for binary classification.
The output is a clean `cleaned_data.csv` with exactly two columns: `answers` and `label`.

- `label = 0` → human-written answer  
- `label = 1` → ChatGPT-generated answer

**Tools used:** pandas, ast — no NLP libraries.

---
## Step 1 — Load and Inspect

We load the raw CSV and do a first-pass inspection: shape, null counts, duplicate counts,
and a head preview. This confirms the file loaded correctly and gives us a baseline
before any transformation.

In [ ]:
import pandas as pd
import ast

df = pd.read_csv('hc3_all.csv')

print('=== Raw Dataset Inspection ===')
print(f'Shape: {df.shape}')
print(f'Columns: {df.columns.tolist()}')
print()
print('--- Null counts per column ---')
print(df.isnull().sum())
print()
print(f'Duplicate rows: {df.duplicated().sum()}')
print()
print('--- First 5 rows ---')
df.head()

---
## Step 2 — Drop Irrelevant Columns

We drop `id`, `question`, `source`, and any auto-generated `Unnamed` columns that pandas
adds when a CSV has a trailing comma or index column saved by mistake.
Only `human_answers` and `chatgpt_answers` are needed downstream.

In [ ]:
cols_to_drop = [col for col in df.columns
                if 'Unnamed' in col or col in ['id', 'question', 'source']]

print(f'Dropping columns: {cols_to_drop}')
df.drop(columns=cols_to_drop, inplace=True)

print(f'Remaining columns: {df.columns.tolist()}')
print(f'Shape after drop: {df.shape}')
df.head()

---
## Step 3 — Build Human Dataframe (Explode)

The `human_answers` column stores multiple answers per question as a **serialised Python list
string**, e.g. `"['Answer one', 'Answer two']"`. We must:

1. Parse each string into a real Python list using `ast.literal_eval()` (safe, unlike `eval()`)
2. Fall back gracefully if parsing fails — wrap the raw string in a list so no row is silently lost
3. Use `explode()` so every individual human answer becomes its own row
4. Add `label = 0` to mark these as human-written

Without exploding, each row would contain a whole list, making character-length comparisons
meaningless and inflating the apparent length of human answers vs. AI answers.

In [ ]:
def safe_parse(val):
    """Parse a serialised list string with ast.literal_eval; fall back to a single-item list."""
    try:
        result = ast.literal_eval(val)
        if isinstance(result, list):
            return result
        return [str(result)]
    except (ValueError, SyntaxError):
        return [str(val)]

human_df = df[['human_answers']].copy()
print(f'Human df shape before explode: {human_df.shape}')

human_df['human_answers'] = human_df['human_answers'].apply(safe_parse)
human_df = human_df.explode('human_answers')
human_df.rename(columns={'human_answers': 'answers'}, inplace=True)
human_df['label'] = 0
human_df = human_df.reset_index(drop=True)

print(f'Human df shape after explode:  {human_df.shape}')
print(f'Sample human answers:')
print(human_df['answers'].head(3).tolist())

---
## Step 4 — Build AI Dataframe (Explode)

Same approach for `chatgpt_answers`. Although the dataset typically stores a single AI answer
per row, it is still stored as a serialised list string, and some rows may contain multiple
AI answers. We parse and explode for consistency and to avoid any list objects leaking through.

In [ ]:
ai_df = df[['chatgpt_answers']].copy()
print(f'AI df shape before explode: {ai_df.shape}')

ai_df['chatgpt_answers'] = ai_df['chatgpt_answers'].apply(safe_parse)
ai_df = ai_df.explode('chatgpt_answers')
ai_df.rename(columns={'chatgpt_answers': 'answers'}, inplace=True)
ai_df['label'] = 1
ai_df = ai_df.reset_index(drop=True)

print(f'AI df shape after explode:  {ai_df.shape}')
print(f'Sample AI answers:')
print(ai_df['answers'].head(3).tolist())

---
## Step 5 — Combine and Clean

We concatenate the human and AI dataframes, then clean up:

- **Drop duplicates** — identical answer text with the same label is redundant
- **Drop nulls and empty strings** — any row where `answers` is NaN or blank after stripping
  would break downstream vectorisation

We print shape at each step to confirm nothing was lost unexpectedly, and check label balance.

In [ ]:
df_combined = pd.concat([human_df, ai_df], ignore_index=True)
print(f'Shape after concat: {df_combined.shape}')

# Drop duplicates
before_dedup = df_combined.shape[0]
df_combined.drop_duplicates(inplace=True)
df_combined.reset_index(drop=True, inplace=True)
after_dedup = df_combined.shape[0]
print(f'Shape after drop_duplicates: {df_combined.shape}  (dropped {before_dedup - after_dedup} rows)')

# Drop nulls
before_null = df_combined.shape[0]
df_combined.dropna(subset=['answers'], inplace=True)
after_null = df_combined.shape[0]
print(f'Shape after dropna: {df_combined.shape}  (dropped {before_null - after_null} null rows)')

# Drop empty strings
before_empty = df_combined.shape[0]
df_combined = df_combined[df_combined['answers'].str.strip() != '']
df_combined.reset_index(drop=True, inplace=True)
after_empty = df_combined.shape[0]
print(f'Shape after empty-string drop: {df_combined.shape}  (dropped {before_empty - after_empty} rows)')

print()
print('--- Label distribution ---')
print(df_combined['label'].value_counts())

---
## Step 6 — Lowercase and Basic Clean

We apply minimal text normalisation:
- Convert all answers to lowercase
- Strip leading/trailing whitespace

**No further cleaning** (no punctuation removal, no stopwords, no stemming).
The downstream TF-IDF + logistic regression pipeline handles the text as-is.

In [ ]:
df_combined['answers'] = df_combined['answers'].str.lower().str.strip()

print('Lowercasing and whitespace stripping applied.')
print(f'Shape: {df_combined.shape}')
print()
print('Sample after normalisation:')
print(df_combined['answers'].head(3).tolist())

---
## Step 7 — Save Output

We assert that every value in `answers` is a plain string (not a list) — this confirms the
explode worked correctly. Then we save `cleaned_data.csv` with exactly two columns:
`answers` and `label`.

In [ ]:
# Enforce only the two required columns
df_final = df_combined[['answers', 'label']].copy()

# Safety assertion: every answer must be a plain string, not a list
assert df_final['answers'].apply(lambda x: isinstance(x, str)).all(), \
    'ASSERTION FAILED: some answers are not plain strings — explode may not have worked!'
print('Assertion passed: all answers are plain strings.')

# Save
df_final.to_csv('cleaned_data.csv', index=False)
print(f'Saved cleaned_data.csv with shape: {df_final.shape}')
print(f'Columns: {df_final.columns.tolist()}')
print()
print('--- Head ---')
print(df_final.head())
print()
print('--- Tail ---')
print(df_final.tail())
print()
print(f'Final shape: {df_final.shape}')

---
## Step 8 — Sanity Check

A final audit of the cleaned dataset to confirm:

- Total rows and label balance
- Average answer length by label — **if human answers are still ~3× longer than AI answers,
  the explode did not work correctly** (because un-exploded human rows would still contain
  full list strings with multiple answers concatenated)
- Count and removal of suspiciously short answers (< 20 characters), which are likely
  parsing artefacts or empty placeholders

In [ ]:
df_check = pd.read_csv('cleaned_data.csv')

print('===============================')
print('       FINAL SANITY CHECK      ')
print('===============================')

print(f'\nTotal rows: {len(df_check)}')

# Label distribution
print('\n--- Label distribution (counts) ---')
counts = df_check['label'].value_counts().sort_index()
print(counts)
print('\n--- Label distribution (percentages) ---')
print((counts / len(df_check) * 100).round(2).astype(str) + '%')

# Source distribution note
print('\n--- Source distribution ---')
print('(source column was dropped in Step 2; not available for distribution check)')

# Average character length by label
df_check['char_len'] = df_check['answers'].str.len()
print('\n--- Average character length by label ---')
avg_len = df_check.groupby('label')['char_len'].mean().round(1)
print(avg_len.rename({0: 'Human (label=0)', 1: 'AI (label=1)'})) 
ratio = avg_len[0] / avg_len[1] if avg_len[1] > 0 else float('inf')
print(f'Ratio human/AI: {ratio:.2f}x')
if ratio > 2.5:
    print('WARNING: Human answers are still much longer than AI answers.')
    print('         This suggests explode() may not have worked correctly.')
else:
    print('OK: Length ratio looks reasonable — explode appears to have worked.')

# Short answer audit
short_mask = df_check['answers'].str.strip().str.len() < 20
n_short = short_mask.sum()
print(f'\n--- Answers shorter than 20 characters: {n_short} ---')
if n_short > 0:
    print('Examples:')
    print(df_check.loc[short_mask, 'answers'].head(10).tolist())

# Drop short answers
before_short = len(df_check)
df_check = df_check[~short_mask].reset_index(drop=True)
after_short = len(df_check)
print(f'Dropped {before_short - after_short} rows with answers < 20 chars.')
print(f'Final shape after short-answer removal: {df_check.shape}')

# Overwrite cleaned_data.csv with the truly final version
df_check[['answers', 'label']].to_csv('cleaned_data.csv', index=False)
print('\ncleaned_data.csv overwritten with final clean version.')
print('\n===============================')
print('      PREPROCESSING COMPLETE   ')
print('===============================')